# Milestone 1: Data Exploration & Retrieval

This notebook covers:
1. Data download and exploration using DuckDB
2. Corpus preprocessing and parquet export
3. Building and testing BM25, semantic, and hybrid retrievers
4. Qualitative evaluation across a query set

In [ ]:
import sys
import duckdb
import pandas as pd
from pathlib import Pathddd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
INDICES_DIR = PROJECT_ROOT / "indices"

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
INDICES_DIR.mkdir(parents=True, exist_ok=True)

CATEGORY = "All_Beauty"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
META_URL = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"

con = duckdb.connect()

## 1. Data Download & Exploration

Using DuckDB to download and explore the All Beauty metadata directly from the McAuley Lab URL.

In [ ]:
# Preview first 5 records directly from URL (no download needed)
head_meta = con.execute(f"SELECT * FROM read_json_auto('{META_URL}') LIMIT 5").df()
head_meta

In [ ]:
# Count total products (reads from URL)
count_df = con.execute(f"SELECT COUNT(*) AS total FROM read_json_auto('{META_URL}')").df()
print(f"Total products: {count_df['total'][0]:,}")

In [ ]:
# Download full metadata and save as parquet locally
con.execute(f"""
    COPY (SELECT * FROM read_json_auto('{META_URL}'))
    TO '{DATA_RAW / "meta_All_Beauty.parquet"}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")
print(f"Saved to {DATA_RAW / 'meta_All_Beauty.parquet'}")

In [ ]:
# Now work from local parquet (fast)
meta_df = con.execute(f"""
    SELECT * FROM read_parquet('{DATA_RAW / "meta_All_Beauty.parquet"}')
""").df()

print(f"Shape: {meta_df.shape}")
print(f"\nColumns: {list(meta_df.columns)}")
print(f"\nNull counts:")
print(meta_df.isnull().sum())

In [ ]:
# Inspect a few sample records
for i in range(3):
    print(f"\n--- Product {i+1} ---")
    for key, value in meta_df.iloc[i].items():
        display_val = str(value)[:200] if isinstance(value, (str, list)) else value
        print(f"  {key}: {display_val}")

## 2. Field Selection & Preprocessing

**Fields selected for retrieval:**
- `title` — primary product identifier
- `description` — detailed product information
- `features` — bullet-point product attributes

**Rationale:** These three fields contain the richest text for matching user queries. Price, rating, and images are kept as metadata for display but not included in the search text.

**Preprocessing decisions:**
- Concatenate title + description + features into a single `text` field per product
- Skip products with no text content
- For BM25: tokenize with lowercase + punctuation removal + whitespace split
- For semantic search: use raw concatenated text (model handles tokenization)

In [ ]:
from src.utils import build_text

records = []
for _, row in meta_df.iterrows():
    product = row.to_dict()
    text = build_text(product)
    if not text.strip():
        continue
    records.append({
        "parent_asin": product.get("parent_asin", ""),
        "title": product.get("title", ""),
        "text": text,
        "price": product.get("price"),
        "average_rating": product.get("average_rating"),
    })

corpus_df = pd.DataFrame(records)
corpus_df.to_parquet(DATA_PROCESSED / "product_corpus.parquet", compression="zstd", index=False)
print(f"Corpus size: {len(corpus_df):,} products")
print(f"Saved to data/processed/product_corpus.parquet")
corpus_df.head()

In [ ]:
print("Price statistics:")
print(corpus_df["price"].describe())
print(f"\nRating statistics:")
print(corpus_df["average_rating"].describe())
print(f"\nText length statistics:")
corpus_df["text_len"] = corpus_df["text"].str.len()
print(corpus_df["text_len"].describe())

## 3. BM25 Retriever

Building a keyword-based retrieval system using BM25 (Okapi BM25).

In [ ]:
from src.bm25 import BM25Retriever
from src.utils import load_corpus

corpus = load_corpus(str(DATA_PROCESSED / "product_corpus.parquet"))

bm25 = BM25Retriever()
bm25.build_index(corpus)
bm25.save(str(INDICES_DIR / "bm25_index.pkl"))
print("BM25 index built and saved.")

results = bm25.search("vitamin C serum", top_k=5)
print("\nTest query: 'vitamin C serum'")
for i, r in enumerate(results):
    print(f"  {i+1}. {r['title']} (score: {r['score']:.4f})")

## 4. Semantic Retriever

Building a semantic search system using sentence-transformers (all-MiniLM-L6-v2) and FAISS.

*Will completes this section.*

## 5. Hybrid Retriever

Combining BM25 and semantic scores with weighted linear combination.

*Will completes this section.*

## 6. Qualitative Evaluation

Running all three methods on our evaluation query set and comparing results.

*Will completes this section.*